In [ ]:
import copy

In [ ]:
def hrs_min_sec(sec_val):
    hours=str(sec_val//(60**2))
    minutes=str((sec_val//60)%60)
    seconds=str(round(sec_val%60,0))
    if sec_val//(60**2)!=0:
        return(hours+' hrs '+minutes+' min '+seconds+' sec')
    if (sec_val//60)%60!=0:
        return(minutes+' min '+seconds+' sec')
    return(seconds+' sec')

In [ ]:
def find_cochain_basis(ss):
    '''args: ss (spanning set), a list of cochains of the same homogeneous degree
       Returns: A list of cochains which are a basis for the subspace spanned by ss'''
    if len(ss)==0: return []
    basis_set=set()
    for c in ss:
        for base_elt in c.coeff_dict:
            basis_set.add(base_elt)
    B=list(basis_set)
    
    M=zeros(len(ss),len(B))
    for i in range(len(ss)):
        set_row(M,i,coordinatize_cochain_in_basis(ss[i],B))
    M=M.rref()[0]
    result=[list(M.row(i)) for i in range(shape(M)[0]) 
            if list(M.row(i))!=[0]*len(M.row(i))]   
    basis_cochains=[cochain({A:1},ss[0].parent) for A in B]
    return([coords_to_lin_comb(A,basis_cochains) for A in result])

In [ ]:
def set_row(mat,rowNum,row):
    if type(row)==type(zeros(3,3)):
        rowList=list(row)
    else: 
        if type(row)==type([0]):
            rowList=row
        else: print('setRow error: arg row must be either matrix or list')
    if len(rowList)!=len(mat.row(0)):
        print('setRow error: mat.row() and row have differing lengths')
        return None
    for i in range(len(rowList)):
        mat[rowNum,i]=rowList[i]
        
def set_col(mat,colNum,col):
    if type(col)==type(zeros(2,2)):
        colList=list(col)
    else:
        if type(col)==type([0]):
            colList=col
        else: print('SetCol error: arg col must be either matrix or list')
            
    if len(colList)!=len(mat.col(0)):
        print('SetCol error: mat.col() and col have differing lengths')
        return None
    for i in range(len(colList)):
        mat[i,colNum]=colList[i]

In [ ]:
def coords_to_lin_comb(basis,coords):
    '''args: basis, a list of symbols, and coords, a vector of the same length
       Returns: A LinComb corresponding to the vector coords
       NOTE: basis must be a list of symbols'''
    
    # Check if the lengths are the same:
    if len(basis)!=len(coords):
        print('coords_to_lin_comb error: |basis| and |coords| have different lengths')
        return None
    
    result=0
    for i in range(len(basis)):
        result=result+coords[i]*basis[i]
    return result

In [ ]:
def coordinatize_cochain_in_basis(c,basis):
    '''c: a cochain object of homogeneous degree
       basis: a collection of str tuples representing elementary cochains
       returns the vector representation of c w.r.t basis as a list'''
    result=[0]*len(basis)
    for key in c.coeff_dict:
        if key in basis: result[basis.index(key)]=c.coeff_dict[key]
        else: print('coordinatize_cochain_in_basis error: cochain component',key,'not in basis')
    return result

In [ ]:
def convert_T_symb_elt_to_cochain(se):
    '''se: a T_symb_elt object or a T_symb_basis object
       returns: a degree 0 cochain object corresponding to se'''
    if se==0: return cochain({},se.heis_dim)
    return cochain({(str(T_symb_basis[i]),):se.vec_rep[i] 
                    for i in range(len(T_symb_basis))},se.heis_dim)

In [ ]:
def permutation_sign(it_1,it_2):
    '''tuple_1, tuple_2: iterables containing the same elements
       returns: the sign of the permutation taking it_1 to it_2'''
    cnt=0
    for i in range(len(it_1)):
        for j in range(i+1,len(it_1)):
            if it_2.index(it_1[j])<it_2.index(it_1[i]):
                cnt+=1
    return (-1)**cnt

In [ ]:
def remove_antisymm_zeros(coeff_dict):
    '''coeff_dict: a coeff_dict for an exterior vector or cochain
       result: coeff_dict, but with keys like (e1,e2,e1) removed'''
    result_dict=copy.copy(coeff_dict)
    for key in list(result_dict):
        if len(set(key))!=len(key):
            result_dict.pop(key)
    return result_dict

In [ ]:
def remove_antisymm_zeros_cochains(coeff_dict):
    '''coeff_dict: a coeff_dict for an exterior vector or cochain
       result: coeff_dict, but with keys like (e1,e1,e2) removed,
       leaving keys like (e1,e2,e1) representing nontrivial cochains'''
    
    result_dict=copy.copy(coeff_dict)
    for key in list(result_dict):
        if len(set(key[0:len(key)-1]))!=len(key)-1:
            result_dict.pop(key)
    return result_dict

In [ ]:
def merge_coeff_dicts(dict1,dict2):
    result={}
    for key in set(dict1.keys()).union(set(dict2.keys())):
        coeff=0
        if key in dict1:
            coeff+=dict1[key]
        if key in dict2:
            coeff+=dict2[key]
        result[key]=coeff
    return remove_zeros(result)

In [ ]:
def remove_zeros(coeff_dict):
    '''coeff_dict: a dict with integer values
       returns: a copy of coeff_dict with all keys of value 0 removed'''
    return{A:coeff_dict[A] for A in coeff_dict if coeff_dict[A]!=0}

In [ ]:
def str_from_coeff_dict(coeff_dict):
    '''coeff_dict: a dict with keys that are printable objects
                   or tuples of printable objects and integer values
       returns: a string representing the dict'''
    coeff_dict=remove_zeros(coeff_dict)
    if coeff_dict=={}: return '0'
    key_list=list(coeff_dict.keys())
    result=''

    for key in key_list:
        if type(key)==tuple:
            key_str='('+','.join([str(A) for A in key])+')'
        else: key_str=str(key)
        if coeff_dict[key]==1:
            result+=' + '+key_str
        elif coeff_dict[key]==-1:
            result+=' - '+key_str
        elif coeff_dict[key]!=0:
            result+=' + '+str(coeff_dict[key])+'*'+key_str
    if result[0:3]==' + ':
        return result[3:len(result)]
    return result[1:len(result)]

In [ ]:
def iprod_mat(elts):
    '''elts: an iterable of elements with attribute iprod
       returns: the matrix with (ei.iprod(ej)) as its (i,j)-entry'''
    result=zeros(len(elts))            
    for i in range(len(elts)):
        c1=elts[i]
        for j in range(i,len(elts)):
            c2=elts[j]
            val=c1.iprod(c2)
            result[i,j]=val
            result[j,i]=val
    return result